# How to Use This Notebook

This notebook is designed to be run sequentially from top to bottom without manual intervention. The cells are grouped into numbered steps. Please execute each step in order to run the verification.

- **Step 1: Environment Setup:** Prepares the Kaggle environment, installs dependencies, and verifies TPU access.
- **Step 2: Apply Compatibility Fix:** Downgrades NumPy to prevent known version conflicts.
- **Step 3: Configure Checkpoint Path:** Finds the Llama 3.1 checkpoint dataset and sets the required environment variable.
- **Step 4: Generate Config and Run Verification:** Uses the verified `load_parameters_path` key to generate the complete YAML file and runs the final 1-step training verification. Success is indicated by a "Verification run completed successfully" message.


# Step 1 & 2: Environment Setup

This step prepares the Kaggle environment by:
1.  **Verifying JAX and TPU Access:** Ensures the notebook can see the 8 TPU devices.
2.  **Cloning MaxText:** Clones the `google/maxtext` repository, which contains the training scripts.
3.  **Installing Dependencies:** Installs all Python packages required by MaxText from `requirements.txt`.


In [ ]:
# Cell 1: Complete Environment Setup
import os, sys, platform, subprocess

# --- Part 1: Clone MaxText and Install Dependencies ---
print("Verifying JAX and TPU environment...")
try:
    import jax
    print(f"✅ JAX version: {jax.__version__}")
    print(f"✅ Detected {jax.device_count()} TPU devices.")
except Exception as e:
    print(f"❌ ERROR: JAX/TPU verification failed: {e}")
    raise

print("\\nCloning MaxText repository...")
if not os.path.exists('maxtext'):
    subprocess.run(["git", "clone", "https://github.com/google/maxtext.git"], check=True)
print("✅ MaxText repository cloned.")

# Pin to MaxText commit compatible with JAX 0.4.34
stable_commit_hash = "4651cb3c73de"
print(f"\\nChecking out MaxText commit: {stable_commit_hash}")
subprocess.run(["git", "checkout", stable_commit_hash], check=True, cwd="maxtext", capture_output=True)
print("✅ Git checkout successful.")

print("\\nInstalling dependencies...")
subprocess.run(["apt-get", "update"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "pkg-config"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "maxtext/requirements.txt"], check=True, capture_output=True)
print("✅ Dependencies installed.")

# --- Part 2: Apply NumPy Compatibility Fix ---
print("\\nApplying NumPy compatibility fix...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2"], check=True, capture_output=True)

# Verify the installed version in a clean subprocess
result = subprocess.run([sys.executable, "-c", "import numpy as np; print(np.__version__)"], check=True, capture_output=True, text=True)
numpy_version = result.stdout.strip()
print(f"✅ NumPy version is now: {numpy_version}")
print("\\n✅ Full environment setup is complete.")

# ⚠️ Important: Restart the Kernel Now

Go to the **Run** menu and select **Restart session**. This is required for the NumPy version change to take effect. Do not run the cells below until you have restarted the session.

# Step 3: Configure Checkpoint Path

This step dynamically locates the pre-converted Llama 3.1 MaxText checkpoint within the attached Kaggle Datasets and sets the `MAXTEXT_CHECKPOINT_DIR` environment variable. This makes the checkpoint path available for the final verification step.


In [3]:
import os
from pathlib import Path

dataset_path = Path("/kaggle/input/llama-3-1-8b-maxtext-checkpoint")

print(f"Inspecting dataset directory: {dataset_path}")
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

required_files = ["_CHECKPOINT_METADATA", "items"]
if all((dataset_path / f).exists() for f in required_files):
    checkpoint_dir = dataset_path
    print(f"✅ Checkpoint found in root directory: {checkpoint_dir}")
else:
    raise FileNotFoundError(f"Could not find required checkpoint files in {dataset_path}")

os.environ["MAXTEXT_CHECKPOINT_DIR"] = str(checkpoint_dir)
print(f"✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR={os.environ['MAXTEXT_CHECKPOINT_DIR']}")


Inspecting dataset directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Checkpoint found in root directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR=/kaggle/input/llama-3-1-8b-maxtext-checkpoint


# Step 4: Generate Config and Run Verification

This is the final, fully automated step. It performs the following actions:

1.  **Sets `PYTHONPATH`:** Ensures the MaxText library can be correctly imported.
2.  **Generates YAML:** Creates the `verification_minimal.yml` file using the verified `load_parameters_path` key and the checkpoint path from the previous step.
3.  **Runs Verification:** Executes the MaxText training script as a module (`MaxText.train`) for a single step. 

A successful run will print "✅ Verification run completed successfully." and is the evidence that the entire environment is correctly configured.


In [11]:
import os
import sys
import runpy
from pathlib import Path

# Set environment variable to resolve TensorFlow protobuf conflict
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. Define paths and add MaxText to the Python path for in-process import
maxtext_repo_path = Path.cwd() / "maxtext"
maxtext_pkg_path = maxtext_repo_path / "MaxText" # Path to the actual source files

# Add the repo root for runpy to find the 'MaxText' package
if str(maxtext_repo_path) not in sys.path:
    sys.path.insert(0, str(maxtext_repo_path))
    
# Add the package path for scripts inside to find each other (e.g., train.py importing checkpointing.py)
if str(maxtext_pkg_path) not in sys.path:
    sys.path.insert(0, str(maxtext_pkg_path))

print(f"✅ Added to sys.path: {maxtext_pkg_path}")

# 2. Get checkpoint path and define the verified key
checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run the previous step first.")

verified_checkpoint_key = "load_parameters_path"
print(f"✅ Using verified key '{verified_checkpoint_key}' for checkpoint loading.")

# 3. Generate the complete and correct YAML configuration
config_text = f"""
# Auto-generated configuration for verification run
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1
eval_per_device_batch_size: 1
enable_checkpointing: True
async_checkpointing: False
enable_emergency_checkpoint: False
save_config_to_gcs: False
hardware: 'tpu'
jax_cache_dir: "/kaggle/working/jax_cache"
dataset_type: "synthetic"
expansion_factor_real_data: 1.0
log_period: 100
scan_layers: True
attention: 'dot_product'
attention_type: 'global'
{verified_checkpoint_key}: "{checkpoint_path}"
load_full_state_path: ""
compile_topology: False
compile_topology_num_slices: -1
quantization_local_shard_count: -1
ici_data_parallelism: 1
ici_pipeline_parallelism: 1
ici_fsdp_parallelism: 1
ici_fsdp_transpose_parallelism: 1
ici_sequence_parallelism: 1
ici_tensor_parallelism: 1
ici_expert_parallelism: 1
ici_autoregressive_parallelism: 1

# Data Center Network Parallelism (for across multiple hosts)
dcn_data_parallelism: 1
dcn_pipeline_parallelism: 1
dcn_fsdp_parallelism: 1
dcn_fsdp_transpose_parallelism: 1
dcn_sequence_parallelism: 1
dcn_tensor_parallelism: 1
dcn_expert_parallelism: 1
dcn_autoregressive_parallelism: 1

# JAX/Debug defaults to avoid KeyErrors
jax_debug_log_modules: []
jax_disable_jit: False
jax_enable_x64: False
jax_debug_nans: False
jax_profile_server: ""
profiler: ""

# Required model parameters for llama3.1-8b
model_name: "llama3.1-8b"
num_experts: 1
global_parameter_scale: 1
base_emb_dim: 4096
base_num_query_heads: 32
base_num_kv_heads: 8
base_num_decoder_layers: 32
base_mlp_dim: 14336
head_dim: 128
vocab_size: 128256
max_target_length: 2048
max_prefill_predict_length: 1024
normalization_layer_epsilon: 0.00001
decoder_block: llama2
enable_dropout: False
logits_via_embedding: False
mlp_activations: ['silu', 'linear']
dtype: "bfloat16"
attn_logits_soft_cap: 0
final_logits_soft_cap: 0
rope_max_timescale: 500000

# Training configuration
model_call_mode: ''
gradient_accumulation_steps: 1
remat_policy: 'nothing'
learning_rate: 0.0001
learning_rate_schedule_steps: -1
warmup_steps_fraction: 0.01
cosine_learning_rate_final_fraction: 0.1
gradient_clipping_threshold: 1.0
adam_b1: 0.9
adam_b2: 0.95
adam_eps: 0.00000001
adam_weight_decay: 0.1
init_weights_seed: 0

# Sharding configuration
logical_axis_rules: [
  ['embed', 'mp'],
  ['mlp', 'mp'],
  ['attention', 'mp'],
  ['heads', 'mp']
]
data_sharding: ['data', 'model']
compute_axis_order: '0,1,2,3'
kv_quant_axis: ""
quantize_kvcache: False
"""

config_path = Path("/kaggle/working/verification_minimal.yml")
config_path.write_text(config_text)
print(f"✅ Wrote config to: {config_path}")
print("--- Config Contents ---")
print(config_text)
print("-----------------------")

# 4. Run the verification script IN-PROCESS using runpy
print("\\n🚀 Running 1-step verification (in-process)...")

# Temporarily replace sys.argv for the script
original_argv = sys.argv
try:
    # Set argv for the script to run. The first element is the script name,
    # followed by its arguments.
    sys.argv = ['MaxText/train.py', str(config_path)]
    
    # Run the module. 'run_name="__main__"' makes the script believe it's being
    # executed directly.
    runpy.run_module('MaxText.train', run_name='__main__')
    
    print("\\n✅ Verification run completed successfully.")
except SystemExit as e:
    if e.code == 0:
        print("\\n✅ Verification run completed successfully (SystemExit code 0).")
    else:
        # The script likely failed and called sys.exit()
        print(f"\\n❌ Verification run failed with SystemExit code: {e.code}")
except Exception as e:
    # Catch any other unexpected exceptions
    import traceback
    print(f"\\n❌ An unexpected error occurred: {e}")
    traceback.print_exc()
finally:
    # Always restore the original sys.argv
    sys.argv = original_argv

✅ Added to sys.path: /kaggle/working/maxtext/MaxText
✅ Using verified key 'load_parameters_path' for checkpoint loading.
✅ Wrote config to: /kaggle/working/verification_minimal.yml
--- Config Contents ---

# Auto-generated configuration for verification run
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1
eval_per_device_batch_size: 1
enable_checkpointing: True
async_checkpointing: False
enable_emergency_checkpoint: False
save_config_to_gcs: False
hardware: 'tpu'
jax_cache_dir: "/kaggle/working/jax_cache"
dataset_type: "synthetic"
expansion_factor_real_data: 1.0
log_period: 100
scan_layers: True
attention: 'dot_product'
attention_type: 'global'
load_parameters_path: "/kaggle/input/llama-3-1-8b-maxtext-checkpoint"
load_full_state_path: ""
compile_topology: False
compile_topology_num_slices: -1
quantization_local_shard_count: -1
ici_data_parallelism: 1
ici_pipeline_parallelism: 1
ici_fsdp_parallelism: 1
ici_fsdp_tr

Traceback (most recent call last):
  File "/tmp/ipykernel_1567/2585766565.py", line 153, in <module>
    runpy.run_module('MaxText.train', run_name='__main__')
  File "/usr/local/lib/python3.10/runpy.py", line 227, in run_module
    return _run_code(code, {}, init_globals, run_name, mod_spec)
  File "/usr/local/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/kaggle/working/maxtext/MaxText/train.py", line 990, in <module>
    app.run(main)
  File "/usr/local/lib/python3.10/site-packages/absl/app.py", line 316, in run
    _run_main(main, args)
  File "/usr/local/lib/python3.10/site-packages/absl/app.py", line 261, in _run_main
    sys.exit(main(argv))
  File "/kaggle/working/maxtext/MaxText/train.py", line 955, in main
    pyconfig.initialize(argv)
  File "/kaggle/working/maxtext/MaxText/pyconfig.py", line 770, in initialize
    _config = _HyperParameters(argv, **kwargs)
  File "/kaggle/working/maxtext/MaxText/pyconfig.py", line 355, in __init__
    _